In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import lightgbm as lgb

In [3]:
df2 = pd.read_excel("/content/drive/MyDrive/colab_datasets/elevator_fault_dataset_enhanced_further.xlsx")
df2.head()

,Timestamp,Motor_Current,Motor_Voltage,Vibration_Level,Speed,Cabin_Position,Cabin_Acceleration,Cabin_Deceleration,Door_Operation_Time,Braking_Force,...,Door_Operation_Time_diff,Door_Operation_Time_pct_change,Load_Accumulation_diff,Load_Accumulation_pct_change,Pulley_Wear_Accumulated_diff,Pulley_Wear_Accumulated_pct_change,comp_index,time_since_start_sec,Load_x_Speed,Vib_x_Current
0,2020-01-01 00:00:00,22.289903,241.253222,1.393233,3.519079,19,0.349191,0.358643,5.747832,112.280046,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,2974.276168,31.055037
1,2020-01-01 01:00:00,25.240487,236.078046,0.090303,2.226921,9,0.260127,0.266565,7.552277,142.770141,...,1.804444,0.313935,1078.418306,1.275954,46.828076,0.556469,1,3600,2401.552647,2.279296
2,2020-01-01 02:00:00,32.769471,432.220975,0.932987,2.523211,20,0.291057,0.161099,7.703547,98.016876,...,0.151270,0.020030,2355.043882,1.224287,58.900489,0.449690,2,7200,5942.272466,30.573492
3,2020-01-01 03:00:00,34.913962,645.108627,0.490413,4.910428,20,0.439094,0.214058,8.335525,75.588623,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,3856.049430,17.122260
4,2020-01-01 04:00:00,27.799073,224.451891,0.844176,5.188226,20,0.566116,0.365294,4.717031,95.882924,...,-3.618494,-0.434105,2011.447371,2.561447,77.635576,1.185632,1,3600,10435.844443,23.467313


In [4]:
df2 = df2.fillna(0)

In [ ]:
# Replace invalid values
df2 = df2.replace([np.inf, -np.inf], 0)

# Remove invalid RUL rows
df2 = df2[df2['RUL'] > 0]
df2 = df2[df2['RUL'] < df2['RUL'].quantile(0.995)]

# Log-transform RUL (CRITICAL)
y = np.log1p(df2['RUL'].values)

In [ ]:
drop_cols = [
    'Timestamp',
    'RUL',
    'Fault_State',
    'Fault_Category',
    'Fault_Severity',
    'Temp_Category'
]

X = df2.drop(columns=drop_cols, errors='ignore').values

# Remove zero-only / near-zero columns
non_zero_cols = (X != 0).sum(axis=0) > 10
X = X[:, non_zero_cols]

# Scale features
scaler = RobustScaler()
X = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [ ]:
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
train_loader = DataLoader(RULDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader  = DataLoader(RULDataset(X_test, y_test), batch_size=128, shuffle=False)

In [ ]:
class StableFeatureTransformer(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.embedding = nn.Linear(1, 32)
        self.norm = nn.LayerNorm(32)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=32,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=2
        )

        self.regressor = nn.Sequential(
            nn.Linear(32 * n_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
            nn.Softplus()  # ensures positive output
        )

    def forward(self, x):
        x = x.unsqueeze(-1)          # (B, F, 1)
        x = self.embedding(x)        # (B, F, 32)
        x = self.norm(x)
        x = self.transformer(x)
        x = x.reshape(x.size(0), -1)
        return self.regressor(x).squeeze()

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = StableFeatureTransformer(n_features=X_train.shape[1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

for epoch in range(30):
    model.train()
    epoch_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)

        if torch.isnan(loss):
            print("NaN detected — stopping training")
            break

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {epoch_loss / len(train_loader):.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch 1 | Loss: 1.0269
Epoch 2 | Loss: 0.6324
Epoch 3 | Loss: 0.6242
Epoch 4 | Loss: 0.6178
Epoch 5 | Loss: 0.6178
Epoch 6 | Loss: 0.5821
Epoch 7 | Loss: 0.5670
Epoch 8 | Loss: 0.5732
Epoch 9 | Loss: 0.5577
Epoch 10 | Loss: 0.5716
Epoch 11 | Loss: 0.5377
Epoch 12 | Loss: 0.5455
Epoch 13 | Loss: 0.5421
Epoch 14 | Loss: 0.5339
Epoch 15 | Loss: 0.5301
Epoch 16 | Loss: 0.5248
Epoch 17 | Loss: 0.5291
Epoch 18 | Loss: 0.5207
Epoch 19 | Loss: 0.5102
Epoch 20 | Loss: 0.5086
Epoch 21 | Loss: 0.5148
Epoch 22 | Loss: 0.4950
Epoch 23 | Loss: 0.5037
Epoch 24 | Loss: 0.4975
Epoch 25 | Loss: 0.4888
Epoch 26 | Loss: 0.4906
Epoch 27 | Loss: 0.4837
Epoch 28 | Loss: 0.4876
Epoch 29 | Loss: 0.4822
Epoch 30 | Loss: 0.4860


In [ ]:
model.eval()
with torch.no_grad():
    y_pred_nn = torch.cat([
        model(xb.to(device)).cpu()
        for xb, _ in test_loader
    ]).numpy()

In [ ]:
residuals = y_test - y_pred_nn

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(X_test, residuals)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.051307 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 114981
[LightGBM] [Info] Number of data points in the train set: 8573, number of used features: 492
[LightGBM] [Info] Start training from score 0.176746
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, max_depth=8,
              n_estimators=500, random_state=42, subsample=0.8)

In [ ]:
y_pred_log = y_pred_nn + lgb_model.predict(X_test)

# Inverse log-transform
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE : {mae:.2f}")
print(f"R²  : {r2:.4f}")

RMSE: 504.27
MAE : 401.38
R²  : 0.5385


Trying to push for R^2 > 0.55

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [32]:
# Select only int & float columns
feature_cols = df2.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Remove target columns if present
feature_cols = [c for c in feature_cols if c not in ["RUL"]]

# Convert to float32
df2[feature_cols] = df2[feature_cols].astype("float32")

In [34]:
df2["RUL_delta"] = (
    df2.groupby("Component_ID_Proxy")["RUL"]
       .diff()
       .fillna(0)
       .astype("float32")
)

/tmp/ipython-input-2286702803.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2["RUL_delta"] = (


In [35]:
rul_scaler = MinMaxScaler(feature_range=(-1, 1))
df2["RUL_delta_scaled"] = rul_scaler.fit_transform(df2[["RUL_delta"]])

/tmp/ipython-input-144426560.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df2["RUL_delta_scaled"] = rul_scaler.fit_transform(df2[["RUL_delta"]])


In [37]:
cycles = df2["Component_ID_Proxy"].unique()
np.random.shuffle(cycles)

split = int(0.8 * len(cycles))
train_cycles = cycles[:split]
test_cycles  = cycles[split:]

train_df = df2[df2["Component_ID_Proxy"].isin(train_cycles)].reset_index(drop=True)
test_df  = df2[df2["Component_ID_Proxy"].isin(test_cycles)].reset_index(drop=True)

In [38]:
class WindowedDataset(Dataset):
    def __init__(self, df, feature_cols, target_col, window_size=15):
        self.X, self.y = [], []

        for _, g in df.groupby("Component_ID_Proxy"):
            if len(g) <= window_size:
                continue

            Xg = g[feature_cols].values.astype(np.float32)
            yg = g[target_col].values.astype(np.float32)

            for i in range(len(g) - window_size):
                self.X.append(Xg[i:i+window_size])
                self.y.append(yg[i+window_size])

        self.X = torch.tensor(np.array(self.X))
        self.y = torch.tensor(np.array(self.y))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [40]:
WINDOW_SIZE = 6
BATCH_SIZE = 32

train_dataset = WindowedDataset(
    train_df, feature_cols, "RUL_delta_scaled", WINDOW_SIZE
)
test_dataset = WindowedDataset(
    test_df, feature_cols, "RUL_delta_scaled", WINDOW_SIZE
)

assert len(train_dataset) > 0, "No training windows created"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [41]:
class StableRULModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=64,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1]).squeeze()

In [42]:
model = StableRULModel(n_features=len(feature_cols)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.SmoothL1Loss()

In [43]:
NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"[WINDOWED] Epoch {epoch+1} | Loss: {epoch_loss / len(train_loader):.4f}")

[WINDOWED] Epoch 1 | Loss: 0.0344
[WINDOWED] Epoch 2 | Loss: 0.0301
[WINDOWED] Epoch 3 | Loss: 0.0294
[WINDOWED] Epoch 4 | Loss: 0.0289
[WINDOWED] Epoch 5 | Loss: 0.0288
[WINDOWED] Epoch 6 | Loss: 0.0223
[WINDOWED] Epoch 7 | Loss: 0.0239
[WINDOWED] Epoch 8 | Loss: 0.0188
[WINDOWED] Epoch 9 | Loss: 0.0187
[WINDOWED] Epoch 10 | Loss: 0.0185
[WINDOWED] Epoch 11 | Loss: 0.0179
[WINDOWED] Epoch 12 | Loss: 0.0165
[WINDOWED] Epoch 13 | Loss: 0.0164
[WINDOWED] Epoch 14 | Loss: 0.0170
[WINDOWED] Epoch 15 | Loss: 0.0162
[WINDOWED] Epoch 16 | Loss: 0.0157
[WINDOWED] Epoch 17 | Loss: 0.0156
[WINDOWED] Epoch 18 | Loss: 0.0155
[WINDOWED] Epoch 19 | Loss: 0.0161
[WINDOWED] Epoch 20 | Loss: 0.0160


In [45]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        preds = model(xb).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(yb.numpy())

y_true = np.array(y_true).reshape(-1, 1)
y_pred = np.array(y_pred).reshape(-1, 1)

# inverse transform
y_true_inv = rul_scaler.inverse_transform(y_true).ravel()
y_pred_inv = rul_scaler.inverse_transform(y_pred).ravel()

rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
mae  = mean_absolute_error(y_true_inv, y_pred_inv)
r2   = r2_score(y_true_inv, y_pred_inv)

print("\nWINDOWED RESULTS")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"R²   : {r2:.3f}")


WINDOWED RESULTS
RMSE : 894.352
MAE  : 741.980
R²   : 0.554


Introducing Hybrid Loss and Cumulative RUL reconstruction at inference

In [46]:
df = df2.copy().reset_index(drop=True)

# Compute RUL delta (negative degradation rate)
df["RUL_delta"] = df["RUL"].diff().fillna(0.0)

# Select numeric features only
feature_cols = (
    df.select_dtypes(include=["int", "float"])
      .columns
      .drop(["RUL", "RUL_delta"])
      .tolist()
)

# Convert to float32
df[feature_cols] = df[feature_cols].astype("float32")
df["RUL"] = df["RUL"].astype("float32")
df["RUL_delta"] = df["RUL_delta"].astype("float32")

# Scale RUL_delta
delta_scaler = MinMaxScaler(feature_range=(-1, 1))
df["RUL_delta_scaled"] = delta_scaler.fit_transform(
    df[["RUL_delta"]]
).astype("float32")

In [47]:
split_idx = int(len(df) * 0.8)

df_train = df.iloc[:split_idx].reset_index(drop=True)
df_test  = df.iloc[split_idx:].reset_index(drop=True)

In [48]:
WINDOW_SIZE = 6

class LazyWindowedDataset(Dataset):
    def __init__(self, df, feature_cols, window_size):
        self.df = df
        self.feature_cols = feature_cols
        self.window_size = window_size

    def __len__(self):
        return len(self.df) - self.window_size

    def __getitem__(self, idx):
        X = self.df.iloc[idx:idx+self.window_size][self.feature_cols].values
        delta = self.df.iloc[idx+self.window_size]["RUL_delta_scaled"]
        rul = self.df.iloc[idx+self.window_size]["RUL"]

        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(delta, dtype=torch.float32),
            torch.tensor(rul, dtype=torch.float32)
        )

In [49]:
train_dataset = LazyWindowedDataset(df_train, feature_cols, WINDOW_SIZE)
test_dataset  = LazyWindowedDataset(df_test, feature_cols, WINDOW_SIZE)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [50]:
class StableFeatureTransformer(nn.Module):
    def __init__(self, n_features, d_model=128, nhead=8, num_layers=3):
        super().__init__()

        self.input_proj = nn.Linear(n_features, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.regressor(x).squeeze()

In [51]:
class HybridRULLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.2):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.delta_loss = nn.SmoothL1Loss()
        self.abs_loss = nn.MSELoss()

    def forward(self, delta_pred, delta_true, rul_pred, rul_true):
        l1 = self.delta_loss(delta_pred, delta_true)
        l2 = self.abs_loss(rul_pred, rul_true)
        return self.alpha * l1 + self.beta * l2

In [52]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = StableFeatureTransformer(n_features=len(feature_cols)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = HybridRULLoss(alpha=1.0, beta=0.2)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [54]:
NUM_EPOCHS = 30

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for xb, delta_y, rul_y in train_loader:
        xb = xb.to(device)
        delta_y = delta_y.to(device)
        rul_y = rul_y.to(device)

        optimizer.zero_grad()
        delta_pred = model(xb)

        # reconstruct RUL (physics constraint)
        rul_pred = rul_y - delta_pred

        loss = criterion(delta_pred, delta_y, rul_pred, rul_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"[WINDOWED] Epoch {epoch+1} | Loss: {epoch_loss / len(train_loader):.4f}")

[WINDOWED] Epoch 1 | Loss: 1.7264
[WINDOWED] Epoch 2 | Loss: 1.9941
[WINDOWED] Epoch 3 | Loss: 0.8224
[WINDOWED] Epoch 4 | Loss: 0.9888
[WINDOWED] Epoch 5 | Loss: 0.5433
[WINDOWED] Epoch 6 | Loss: 0.8408
[WINDOWED] Epoch 7 | Loss: 0.3406
[WINDOWED] Epoch 8 | Loss: 0.2605
[WINDOWED] Epoch 9 | Loss: 0.2677
[WINDOWED] Epoch 10 | Loss: 1.9702
[WINDOWED] Epoch 11 | Loss: 0.8030
[WINDOWED] Epoch 12 | Loss: 0.4688
[WINDOWED] Epoch 13 | Loss: 0.2338
[WINDOWED] Epoch 14 | Loss: 0.1959
[WINDOWED] Epoch 15 | Loss: 0.1598
[WINDOWED] Epoch 16 | Loss: 0.1366
[WINDOWED] Epoch 17 | Loss: 0.1034
[WINDOWED] Epoch 18 | Loss: 2.0116
[WINDOWED] Epoch 19 | Loss: 0.5914
[WINDOWED] Epoch 20 | Loss: 0.2093
[WINDOWED] Epoch 21 | Loss: 0.3293
[WINDOWED] Epoch 22 | Loss: 0.1162
[WINDOWED] Epoch 23 | Loss: 0.1404
[WINDOWED] Epoch 24 | Loss: 0.1034
[WINDOWED] Epoch 25 | Loss: 0.0853
[WINDOWED] Epoch 26 | Loss: 0.3724
[WINDOWED] Epoch 27 | Loss: 0.0836
[WINDOWED] Epoch 28 | Loss: 0.0572
[WINDOWED] Epoch 29 | Loss: 0

In [60]:
model.eval()

delta_preds = []
rul_anchor = []
rul_true = []

with torch.no_grad():
    for xb, delta_y, rul_y in test_loader:
        xb = xb.to(device)

        delta_p = model(xb).squeeze().cpu().numpy()

        delta_preds.append(delta_p)
        rul_anchor.append((rul_y - delta_y).numpy())  # anchor RUL
        rul_true.append(rul_y.numpy())

delta_preds = np.concatenate(delta_preds).reshape(-1, 1)
rul_anchor = np.concatenate(rul_anchor)
rul_true = np.concatenate(rul_true)

# Inverse-scale delta predictions
delta_preds = delta_scaler.inverse_transform(delta_preds).flatten()

# Proper reconstruction (NO cumulative sum)
rul_pred = rul_anchor + delta_preds

# Metrics
rmse = np.sqrt(mean_squared_error(rul_true, rul_pred))
mae  = mean_absolute_error(rul_true, rul_pred)
r2   = r2_score(rul_true, rul_pred)

print("\nFINAL WINDOWED RESULTS (CORRECT)")
print("RMSE :", round(rmse, 3))
print("MAE  :", round(mae, 3))
print("R²   :", round(r2, 3))


FINAL WINDOWED RESULTS (CORRECT)
RMSE : 719.496
MAE  : 636.388
R²   : 0.098
